# P2 Strict: A비율 형성모형

이 노트북은 P4 strict-clean/manual-approved 계약을 SSOT로 사용해 2024 학과 단위 A비율 형성모형을 실행한다.

- strict D08, sample registry, sample membership, manual feature registry, target denylist는 P4 readiness 산출물에서 읽는다.
- 기존 D08·P4·P5 산출물은 수정하지 않는다.
- target, 취업·진학 outcome, C24/GOMS context, ID/name/metadata leakage 변수는 모델 feature로 사용하지 않는다.
- P2-S와 P2-Q는 각자 표본 안에서만 중첩모형을 비교한다.

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import platform
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import patsy
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("default")

RANDOM_STATE = 3085
KEY = "outcome_row_id"
TARGET = "a_rate_pct"
OUT_ROOT_NAME = "p2_grade_formation_v1_strict"
DEV_SPLITS = ["train", "validation", "val"]
VALIDATION_SPLITS = ["validation", "val"]

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    marker = Path("workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_V4_ACTIVE_INPUT_AUDIT.csv")
    for path in [cur, *cur.parents]:
        if (path / marker).exists():
            return path
    raise RuntimeError("repo root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUT_ROOT = PROJECT_ROOT / "workbook/p2/p2_4" / OUT_ROOT_NAME
NOTEBOOK_PATH = OUTPUT_ROOT / "p2_grade_formation_strict.ipynb"
ARTIFACTS_DIR = OUTPUT_ROOT / "artifacts"
DATA_DIR = OUTPUT_ROOT / "data"
FIGURES_DIR = OUTPUT_ROOT / "figures"
QA_DIR = OUTPUT_ROOT / "qa"
REPORTS_DIR = OUTPUT_ROOT / "reports"
LOGS_DIR = OUTPUT_ROOT / "logs"

for path in [OUTPUT_ROOT, ARTIFACTS_DIR, DATA_DIR, FIGURES_DIR, QA_DIR, REPORTS_DIR, LOGS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def rel(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)

def parse_shape(value):
    if isinstance(value, (list, tuple)):
        return tuple(value)
    if pd.isna(value):
        return None
    text = str(value).strip().strip("[]()")
    parts = [p.strip() for p in text.split(",") if p.strip() and p.strip().lower() != "none"]
    return tuple(int(float(p)) for p in parts) if parts else None

def file_shape(path: Path):
    if path.suffix == ".parquet":
        return tuple(pd.read_parquet(path).shape)
    if path.suffix.lower() == ".csv":
        try:
            return tuple(pd.read_csv(path).shape)
        except pd.errors.EmptyDataError:
            return (0, 0)
    if path.suffix.lower() == ".json":
        payload = json.loads(path.read_text(encoding="utf-8"))
        return (len(payload), None)
    return None

def as_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})

def git_commit() -> str:
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
    except Exception as exc:
        return f"UNAVAILABLE: {type(exc).__name__}"

EXECUTION_ENV = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "statsmodels": sm.__version__,
    "working_directory": str(PROJECT_ROOT),
    "git_commit": git_commit(),
    "execution_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "notebook_path": str(NOTEBOOK_PATH),
    "output_root": str(OUTPUT_ROOT),
}
(LOGS_DIR / "execution_environment.json").write_text(json.dumps(EXECUTION_ENV, ensure_ascii=False, indent=2), encoding="utf-8")
(LOGS_DIR / "execution_stdout.log").write_text("See executed notebook outputs.\n", encoding="utf-8")
(LOGS_DIR / "execution_stderr.log").write_text("", encoding="utf-8")
print(json.dumps(EXECUTION_ENV, ensure_ascii=False, indent=2))

{
  "python": "3.12.3",
  "platform": "Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.39",
  "pandas": "3.0.3",
  "numpy": "2.5.0",
  "statsmodels": "0.14.6",
  "working_directory": "/home/sieg/projects-wsl/SBS_dataScience",
  "git_commit": "5b1a3d54266d881a839ad9a3cec750da66e94bc7",
  "execution_timestamp_utc": "2026-07-13T06:29:22.053715+00:00",
  "notebook_path": "/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p2_grade_formation_v1_strict/p2_grade_formation_strict.ipynb",
  "output_root": "/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p2_grade_formation_v1_strict"
}


In [2]:
P4_ROOT = PROJECT_ROOT / "workbook/p2/p2_4/p4_modeling_readiness_v4"
P4_STATUS_PATH = P4_ROOT / "reports/P4_FINAL_MODELING_READINESS.json"
P4_INPUT_AUDIT_PATH = P4_ROOT / "qa/P4_V4_ACTIVE_INPUT_AUDIT.csv"

p4_status = json.loads(P4_STATUS_PATH.read_text(encoding="utf-8"))
p4_input_audit = pd.read_csv(P4_INPUT_AUDIT_PATH)

def input_record(label: str) -> dict:
    hit = p4_input_audit.loc[p4_input_audit["label"].eq(label)]
    if hit.empty:
        raise RuntimeError(f"missing P4 input label: {label}")
    return hit.iloc[0].to_dict()

strict_d08_record = input_record("strict_mart_department_model_base_2024")
manual_feature_record = input_record("manual_approved_feature_registry")
strict_target_counts_record = input_record("strict_target_sample_counts")

INPUTS = {
    "strict_d08": Path(strict_d08_record["path"]),
    "phase_sample_registry": P4_ROOT / "artifacts/P4_PHASE_SAMPLE_REGISTRY_FINAL.csv",
    "phase_sample_membership": P4_ROOT / "data/P4_PHASE_SAMPLE_MEMBERSHIP_FINAL.parquet",
    "manual_feature_registry": Path(manual_feature_record["path"]),
    "target_denylist": P4_ROOT / "qa/P4_TARGET_SPECIFIC_DENYLIST_FINAL.csv",
    "p4_readiness_status": P4_STATUS_PATH,
    "strict_target_counts": Path(strict_target_counts_record["path"]),
}

expected_strict_shape = tuple(p4_status["active_d08_shape"])
expected_strict_sha = p4_status["active_d08_sha256"]
blockers = []
input_rows = []
for name, path in INPUTS.items():
    exists = path.exists()
    shape = file_shape(path) if exists else None
    sha = sha256_file(path) if exists else None
    input_rows.append({"input": name, "path": rel(path), "exists": exists, "shape": shape, "sha256": sha})
    if not exists:
        blockers.append(f"missing input: {name}")

if file_shape(INPUTS["strict_d08"]) != expected_strict_shape:
    blockers.append("strict D08 shape mismatch vs P4 readiness")
if sha256_file(INPUTS["strict_d08"]) != expected_strict_sha:
    blockers.append("strict D08 hash mismatch vs P4 readiness")
if expected_strict_shape != (7592, 151):
    blockers.append(f"unexpected strict D08 shape in P4 status: {expected_strict_shape}")
if expected_strict_sha != "5f56e375fd1c0474a5e55652859ae007e2f45becd6d3350ee4c82e21fab8df9b":
    blockers.append("unexpected strict D08 sha in P4 status")

df_input_contract = pd.DataFrame(input_rows)
display(df_input_contract)
df_input_contract.to_csv(QA_DIR / "P2_INPUT_CONTRACT.csv", index=False)

if blockers:
    P2_INPUT_STATUS = "BLOCKED_HASH"
    raise RuntimeError("; ".join(blockers))
P2_INPUT_STATUS = "READY"
print("P2_INPUT_STATUS =", P2_INPUT_STATUS)

,input,path,exists,shape,sha256
0,strict_d08,workbook/p2/p2_4/source_eda/strict_clean_v1/ma...,True,"(7592, 151)",5f56e375fd1c0474a5e55652859ae007e2f45becd6d335...
1,phase_sample_registry,workbook/p2/p2_4/p4_modeling_readiness_v4/arti...,True,"(12, 12)",d518bab363399a31037590a2cb285f2a76e88318f5d25b...
2,phase_sample_membership,workbook/p2/p2_4/p4_modeling_readiness_v4/data...,True,"(7592, 21)",c6504fd04c918ce33439462f7b277aa8ae4363f185ef93...
3,manual_feature_registry,workbook/p2/p2_4/source_eda/human_handoff_pack...,True,"(198, 11)",2cdb7797c4619c625fc6a171710970b7691446f4d62cae...
4,target_denylist,workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P...,True,"(16, 3)",97435cb3a6b20b490ce2ce7e65160414517aee5b472f8e...
5,p4_readiness_status,workbook/p2/p2_4/p4_modeling_readiness_v4/repo...,True,"(29, None)",fd8c33ddd2999a59f82a9c24f08a9db9073112c9aad517...
6,strict_target_counts,workbook/p2/p2_4/source_eda/strict_clean_v1/st...,True,"(6, 7)",dde484367e1849b34a28b17e99381f59c04c94fb549d78...


P2_INPUT_STATUS = READY


In [3]:
df_d08 = pd.read_parquet(INPUTS["strict_d08"])
df_sample_registry = pd.read_csv(INPUTS["phase_sample_registry"])
df_membership = pd.read_parquet(INPUTS["phase_sample_membership"])
df_manual_raw = pd.read_csv(INPUTS["manual_feature_registry"])
df_target_denylist = pd.read_csv(INPUTS["target_denylist"])
df_strict_counts = pd.read_csv(INPUTS["strict_target_counts"])

if not df_d08[KEY].is_unique:
    raise RuntimeError("strict D08 key is not unique")
if not df_membership[KEY].is_unique:
    raise RuntimeError("membership key is not unique")

df_base = df_d08.merge(df_membership, on=KEY, how="left", validate="one_to_one", suffixes=("", "_membership"))
sample_flags = [c for c in df_base.columns if c.startswith("sample_")]
if df_base[sample_flags].isna().all(axis=1).any():
    raise RuntimeError("missing membership rows")

def sample_registry_row(sample_id: str) -> dict:
    hit = df_sample_registry.loc[df_sample_registry["sample_id"].eq(sample_id)]
    if hit.empty:
        raise RuntimeError(f"sample registry missing {sample_id}")
    return hit.iloc[0].to_dict()

def build_sample(sample_id: str) -> pd.DataFrame:
    flag = f"sample_{sample_id}"
    sample = df_base.loc[as_bool(df_base[flag])].copy()
    sample["p2_sample_id"] = sample_id
    return sample

df_p2_structure = build_sample("P2_STRUCTURE_READY")
df_p2_selectivity = build_sample("P2_SELECTIVITY_READY")

def sample_audit_row(frame: pd.DataFrame, sample_id: str) -> dict:
    reg = sample_registry_row(sample_id)
    return {
        "sample_id": sample_id,
        "row_n": len(frame),
        "registry_row_n": int(reg["row_n"]),
        "school_n": int(frame["school_uid"].nunique()),
        "registry_school_n": int(reg["school_n"]),
        "campus_n": int(frame["campus_uid"].nunique()),
        "registry_campus_n": int(reg["campus_n"]),
        "major7_n": int(frame["major_group_7"].nunique()),
        "registry_major7_n": int(reg["major7_n"]),
        "train_n": int((frame["split"] == "train").sum()),
        "registry_train_n": int(reg["train_n"]),
        "validation_n": int(frame["split"].isin(VALIDATION_SPLITS).sum()),
        "registry_validation_n": int(reg["validation_n"]),
        "test_n": int((frame["split"] == "test").sum()),
        "registry_test_n": int(reg["test_n"]),
        "target_non_null_n": int(frame[TARGET].notna().sum()),
        "row_id_hash": reg["row_id_hash"],
    }

df_sample_audit = pd.DataFrame([
    sample_audit_row(df_p2_structure, "P2_STRUCTURE_READY"),
    sample_audit_row(df_p2_selectivity, "P2_SELECTIVITY_READY"),
])
display(df_sample_audit)
df_sample_audit.to_csv(QA_DIR / "P2_SAMPLE_AUDIT.csv", index=False)

if int(df_strict_counts.set_index("target_candidate").loc[TARGET, "strict_target_keep_n"]) != len(df_p2_structure):
    raise RuntimeError("P2 structure sample does not match strict target count for a_rate_pct")
if not (df_sample_audit["row_n"].eq(df_sample_audit["registry_row_n"]).all()):
    raise RuntimeError("sample row_n mismatch")
if not df_p2_structure[TARGET].dropna().between(0, 100).all():
    raise RuntimeError("target outside 0..100")

DATA_DIR.mkdir(exist_ok=True, parents=True)
df_p2_structure.to_parquet(DATA_DIR / "P2_STRUCTURE_FRAME.parquet", index=False)
df_p2_selectivity.to_parquet(DATA_DIR / "P2_SELECTIVITY_FRAME.parquet", index=False)
P2_SAMPLE_STATUS = "READY"

,sample_id,row_n,registry_row_n,school_n,registry_school_n,campus_n,registry_campus_n,major7_n,registry_major7_n,train_n,registry_train_n,validation_n,registry_validation_n,test_n,registry_test_n,target_non_null_n,row_id_hash
0,P2_STRUCTURE_READY,7592,7592,197,197,261,261,7,7,5534,5534,1168,1168,890,890,7592,2e3fb7557ee894b50542591376211ec84817501cd7b412...
1,P2_SELECTIVITY_READY,3119,3119,146,146,179,179,7,7,2293,2293,514,514,312,312,3119,1a0e36bb24d7e7f894897503bf0baf2c36f9b8710978b9...


In [4]:
EXPECTED_FEATURES = {
    "major_group_7": "S0",
    "school_type": "S0",
    "region_sido": "S0",
    "campus_branch": "S0",
    "credit_forfeit_flag": "POLICY",
    "female_student_share_pct": "B_CORE",
    "international_student_share_pct": "B_CORE",
    "leave_rate_pct": "B_CORE",
    "student_faculty_ratio": "B_CORE",
    "fulltime_faculty_share_pct": "B_CORE",
    "log_enrolled_students": "B_SCALE",
    "log_graduates": "B_SCALE",
    "selectivity_proxy_pct": "Q_PRIMARY",
    "competition_ratio": "Q_PRIMARY",
    "admission_yield_ratio": "Q_SENSITIVITY",
    "admit_per_applicant_ratio": "Q_SENSITIVITY",
}
S0 = ["major_group_7", "school_type", "region_sido", "campus_branch"]
POLICY = ["credit_forfeit_flag"]
B_CORE = [
    "female_student_share_pct",
    "international_student_share_pct",
    "leave_rate_pct",
    "student_faculty_ratio",
    "fulltime_faculty_share_pct",
]
B_SCALE = ["log_enrolled_students", "log_graduates"]
Q_PRIMARY = ["selectivity_proxy_pct", "competition_ratio"]
Q_SENSITIVITY = ["admission_yield_ratio", "admit_per_applicant_ratio"]

manual = df_manual_raw.copy()
manual["manual_approved_p4_use"] = as_bool(manual["manual_approved_p4_use"])
manual_agg = (
    manual.groupby("column", dropna=False)
    .agg(
        manual_approved=("manual_approved_p4_use", "max"),
        dtype=("dtype", "first"),
        strict_global_block_reason=("strict_global_block_reason", lambda s: "|".join(sorted(set(s.dropna().astype(str))))),
        registry_reason=("registry_reason", lambda s: "|".join(sorted(set(s.dropna().astype(str))))),
    )
    .reset_index()
)

if {"target", "blocked_feature"}.issubset(df_target_denylist.columns):
    denylisted = set(
        df_target_denylist.loc[
            df_target_denylist["target"].astype(str).eq(TARGET),
            "blocked_feature",
        ]
        .dropna()
        .astype(str)
    )
else:
    denylisted = set()
target_like_exact = {
    TARGET,
    "cd_rate_pct",
    "f_rate_pct",
    "expected_a_rate",
    "has_employment",
    "has_progression",
} | denylisted

def forbidden_reason(feature: str) -> str:
    lower = feature.lower()
    if feature in target_like_exact or lower.startswith("expected_a_rate") or lower.startswith("grade_residual"):
        return "target_or_grade_signal_leakage"
    if "employment_rate" in lower or "health_employment_rate" in lower:
        return "employment_outcome_leakage"
    if "progression_rate" in lower or "graduate_school_progression_rate" in lower:
        return "progression_outcome_leakage"
    if lower.startswith("ctx24_") or lower.startswith("goms_"):
        return "context_leakage"
    if lower.endswith("_uid") or "name" in lower:
        return "id_or_name"
    if any(token in lower for token in ["match", "review", "quality", "lineage", "source", "sha256", "hash"]):
        return "metadata_leakage"
    if feature in denylisted:
        return "target_specific_denylist"
    return ""

def missing_pct(frame: pd.DataFrame, feature: str) -> float:
    return float(frame[feature].isna().mean()) if feature in frame.columns and len(frame) else np.nan

rows = []
expected_set = set(EXPECTED_FEATURES)
approved_set = set(manual_agg.loc[manual_agg["manual_approved"], "column"])
for feature, block in EXPECTED_FEATURES.items():
    m = manual_agg.loc[manual_agg["column"].eq(feature)]
    exists = feature in df_d08.columns
    approved = bool(m["manual_approved"].iloc[0]) if not m.empty else False
    deny = bool(forbidden_reason(feature))
    if not exists:
        status, reason = "MISSING", "not in strict D08"
    elif deny:
        status, reason = "DENYLISTED", forbidden_reason(feature)
    elif not approved:
        status, reason = "NOT_APPROVED", "manual_approved_p4_use is not True"
    else:
        status, reason = "MATCH", ""
    rows.append(
        {
            "feature": feature,
            "expected_block": block,
            "actual_block": block if exists else "",
            "exists": exists,
            "manual_approved": approved,
            "denylisted": deny,
            "dtype": str(df_d08[feature].dtype) if exists else "",
            "missing_pct_structure": missing_pct(df_p2_structure, feature),
            "missing_pct_selectivity": missing_pct(df_p2_selectivity, feature),
            "status": status,
            "reason": reason,
        }
    )

for feature in sorted(approved_set - expected_set):
    if feature not in df_d08.columns:
        continue
    rows.append(
        {
            "feature": feature,
            "expected_block": "",
            "actual_block": "manual_approved_extra",
            "exists": True,
            "manual_approved": True,
            "denylisted": bool(forbidden_reason(feature)),
            "dtype": str(df_d08[feature].dtype),
            "missing_pct_structure": missing_pct(df_p2_structure, feature),
            "missing_pct_selectivity": missing_pct(df_p2_selectivity, feature),
            "status": "EXTRA",
            "reason": forbidden_reason(feature),
        }
    )

df_feature_contract_diff = pd.DataFrame(rows)
display(df_feature_contract_diff.query("expected_block != ''"))
df_feature_contract_diff.to_csv(QA_DIR / "P2_FEATURE_CONTRACT_DIFF.csv", index=False)

s_required = S0 + B_CORE + B_SCALE + POLICY
q_required = s_required + Q_PRIMARY + Q_SENSITIVITY
bad_status = {"MISSING", "NOT_APPROVED", "DENYLISTED"}
bad_s = df_feature_contract_diff.loc[df_feature_contract_diff["feature"].isin(s_required) & df_feature_contract_diff["status"].isin(bad_status)]
bad_q = df_feature_contract_diff.loc[df_feature_contract_diff["feature"].isin(q_required) & df_feature_contract_diff["status"].isin(bad_status)]
P2_S_FEATURE_STATUS = "READY" if bad_s.empty else "BLOCKED_FEATURE_CONTRACT"
P2_Q_FEATURE_STATUS = "READY" if bad_q.empty else "BLOCKED_FEATURE_CONTRACT"
P2_FEATURE_CONTRACT_STATUS = "READY" if bad_s.empty and bad_q.empty else "READY_WITH_WARNINGS"
print("P2_S_FEATURE_STATUS =", P2_S_FEATURE_STATUS)
print("P2_Q_FEATURE_STATUS =", P2_Q_FEATURE_STATUS)

model_feature_candidates = s_required if P2_S_FEATURE_STATUS == "READY" else []
leakage_violations = [f for f in model_feature_candidates if forbidden_reason(f)]
if leakage_violations:
    raise RuntimeError(f"model feature leakage violation: {leakage_violations}")

,feature,expected_block,actual_block,exists,manual_approved,denylisted,dtype,missing_pct_structure,missing_pct_selectivity,status,reason
0,major_group_7,S0,S0,True,True,False,str,0.000000,0.000000,MATCH,
1,school_type,S0,S0,True,True,False,string,0.000000,0.000000,MATCH,
2,region_sido,S0,S0,True,True,False,string,0.000000,0.000000,MATCH,
3,campus_branch,S0,S0,True,True,False,string,0.000000,0.000000,MATCH,
4,credit_forfeit_flag,POLICY,POLICY,True,True,False,boolean,0.000000,0.000000,MATCH,
5,female_student_share_pct,B_CORE,B_CORE,True,True,False,Float32,0.004347,0.000641,MATCH,
6,international_student_share_pct,B_CORE,B_CORE,True,True,False,Float32,0.004347,0.000641,MATCH,
7,leave_rate_pct,B_CORE,B_CORE,True,True,False,Float32,0.004347,0.000641,MATCH,
8,student_faculty_ratio,B_CORE,B_CORE,True,True,False,Float32,0.210090,0.038153,MATCH,
9,fulltime_faculty_share_pct,B_CORE,B_CORE,True,True,False,Float32,0.210090,0.038153,MATCH,


P2_S_FEATURE_STATUS = READY
P2_Q_FEATURE_STATUS = BLOCKED_FEATURE_CONTRACT


In [5]:
def numeric_describe(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.select_dtypes(include=[np.number]).describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T.reset_index(names="column")

def categorical_describe(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in frame.select_dtypes(exclude=[np.number]).columns:
        vc = frame[col].astype("string").value_counts(dropna=False)
        rows.append({"column": col, "nunique": int(frame[col].nunique(dropna=True)), "top": str(vc.index[0]) if len(vc) else "", "top_n": int(vc.iloc[0]) if len(vc) else 0})
    return pd.DataFrame(rows)

def target_profile(frame: pd.DataFrame, branch: str) -> dict:
    s = pd.to_numeric(frame[TARGET], errors="coerce")
    return {
        "branch": branch,
        "n": int(s.notna().sum()),
        "mean": s.mean(),
        "std": s.std(),
        "min": s.min(),
        "p01": s.quantile(.01),
        "p05": s.quantile(.05),
        "p25": s.quantile(.25),
        "median": s.median(),
        "p75": s.quantile(.75),
        "p95": s.quantile(.95),
        "p99": s.quantile(.99),
        "max": s.max(),
        "zero_ratio": float((s == 0).mean()),
        "hundred_ratio": float((s == 100).mean()),
        "skewness": s.skew(),
    }

for name, frame in [("structure", df_p2_structure), ("selectivity", df_p2_selectivity)]:
    print("=" * 80, name, frame.shape)
    display(frame.head())
    display(frame.tail())
    display(frame.sample(min(10, len(frame)), random_state=RANDOM_STATE))
    frame.info()
    top_missing = frame.isna().sum().sort_values(ascending=False).head(50).reset_index()
    top_missing.columns = ["column", "missing_n"]
    top_missing["missing_pct"] = top_missing["missing_n"] / len(frame)
    major_n = frame.groupby("major_group_7", dropna=False).agg(row_n=(KEY, "size"), school_n=("school_uid", "nunique")).reset_index()
    school_n = frame.groupby("school_uid", dropna=False).agg(row_n=(KEY, "size"), major7_n=("major_group_7", "nunique")).reset_index()
    numeric_describe(frame).to_csv(QA_DIR / f"P2_{name.upper()}_NUMERIC_DESCRIBE.csv", index=False)
    categorical_describe(frame).to_csv(QA_DIR / f"P2_{name.upper()}_CATEGORICAL_DESCRIBE.csv", index=False)
    top_missing.to_csv(QA_DIR / f"P2_{name.upper()}_TOP_MISSING.csv", index=False)
    major_n.to_csv(QA_DIR / f"P2_{name.upper()}_MAJOR7_N.csv", index=False)
    school_n.to_csv(QA_DIR / f"P2_{name.upper()}_SCHOOL_N.csv", index=False)
    display(top_missing.head(20))
    display(major_n)

df_target_profile = pd.DataFrame([target_profile(df_p2_structure, "P2-S"), target_profile(df_p2_selectivity, "P2-Q")])
display(df_target_profile)
df_target_profile.to_csv(ARTIFACTS_DIR / "P2_TARGET_PROFILE.csv", index=False)

df_base["has_selectivity_observed"] = as_bool(df_base["has_selectivity"]) if "has_selectivity" in df_base.columns else df_base["selectivity_proxy_pct"].notna()
selection_rows = []
for col in [TARGET, "log_enrolled_students", "log_graduates", "student_faculty_ratio"]:
    for observed, sub in df_base.groupby("has_selectivity_observed", dropna=False):
        selection_rows.append({"variable": col, "has_selectivity": bool(observed), "n": len(sub), "mean": pd.to_numeric(sub[col], errors="coerce").mean(), "median": pd.to_numeric(sub[col], errors="coerce").median(), "missing_pct": sub[col].isna().mean()})
for col in ["major_group_7", "school_type", "region_sido", "campus_branch"]:
    ct = pd.crosstab(df_base[col], df_base["has_selectivity_observed"], dropna=False, normalize="columns")
    for idx, row in ct.iterrows():
        selection_rows.append({"variable": col, "level": idx, "share_no_selectivity": row.get(False, np.nan), "share_has_selectivity": row.get(True, np.nan)})
df_selection_audit = pd.DataFrame(selection_rows)
display(df_selection_audit.head(40))
df_selection_audit.to_csv(QA_DIR / "P2_SELECTIVITY_SELECTION_AUDIT.csv", index=False)

================================================================================ structure (7592, 172)


,analysis_year,outcome_row_id,school_name_raw,school_name_base_raw,school_name_std,campus_name_raw_x,campus_seq,campus_branch,campus_name_std_x,dept_name_raw,...,sample_P2_SELECTIVITY_READY,sample_P3_STRUCTURE_READY,sample_P3_SELECTIVITY_READY,sample_P4_E_STRUCTURE_READY,sample_P4_P_STRUCTURE_READY,sample_P4_JOINT_STRUCTURE_READY,sample_P4_E_SELECTIVITY_READY,sample_P4_P_SELECTIVITY_READY,sample_P4_JOINT_SELECTIVITY_READY,p2_sample_id
0,2024,OC2024_00001,가야대학교(김해),가야대학교,가야대학교,김해,,,unknown,간호학과,...,True,True,True,True,True,True,True,True,True,P2_STRUCTURE_READY
1,2024,OC2024_00002,가야대학교(김해),가야대학교,가야대학교,김해,,,unknown,경영물류학과,...,False,True,False,True,True,True,False,False,False,P2_STRUCTURE_READY
2,2024,OC2024_00003,가야대학교(김해),가야대학교,가야대학교,김해,,,unknown,경찰소방학과,...,False,True,False,True,True,True,False,False,False,P2_STRUCTURE_READY
3,2024,OC2024_00004,가야대학교(김해),가야대학교,가야대학교,김해,,,unknown,귀금속주얼리학과,...,False,True,False,True,True,True,False,False,False,P2_STRUCTURE_READY
4,2024,OC2024_00005,가야대학교(김해),가야대학교,가야대학교,김해,,,unknown,물리치료학과,...,False,True,False,True,True,True,False,False,False,P2_STRUCTURE_READY


,analysis_year,outcome_row_id,school_name_raw,school_name_base_raw,school_name_std,campus_name_raw_x,campus_seq,campus_branch,campus_name_std_x,dept_name_raw,...,sample_P2_SELECTIVITY_READY,sample_P3_STRUCTURE_READY,sample_P3_SELECTIVITY_READY,sample_P4_E_STRUCTURE_READY,sample_P4_P_STRUCTURE_READY,sample_P4_JOINT_STRUCTURE_READY,sample_P4_E_SELECTIVITY_READY,sample_P4_P_SELECTIVITY_READY,sample_P4_JOINT_SELECTIVITY_READY,p2_sample_id
7587,2024,OC2024_10232,화성의과학대학교,화성의과학대학교,화성의과학대학교,,,,unknown,서비스디자인학과,...,True,True,True,False,False,False,False,False,False,P2_STRUCTURE_READY
7588,2024,OC2024_10235,화성의과학대학교,화성의과학대학교,화성의과학대학교,,,,unknown,스포츠과학부,...,True,True,True,False,False,False,False,False,False,P2_STRUCTURE_READY
7589,2024,OC2024_10236,화성의과학대학교,화성의과학대학교,화성의과학대학교,,,,unknown,스포츠레저학과,...,False,True,False,True,True,True,False,False,False,P2_STRUCTURE_READY
7590,2024,OC2024_10239,화성의과학대학교,화성의과학대학교,화성의과학대학교,,,,unknown,의료심리학과,...,True,True,True,False,False,False,False,False,False,P2_STRUCTURE_READY
7591,2024,OC2024_10240,화성의과학대학교,화성의과학대학교,화성의과학대학교,,,,unknown,의생명과학과,...,True,True,True,False,False,False,False,False,False,P2_STRUCTURE_READY


,analysis_year,outcome_row_id,school_name_raw,school_name_base_raw,school_name_std,campus_name_raw_x,campus_seq,campus_branch,campus_name_std_x,dept_name_raw,...,sample_P2_SELECTIVITY_READY,sample_P3_STRUCTURE_READY,sample_P3_SELECTIVITY_READY,sample_P4_E_STRUCTURE_READY,sample_P4_P_STRUCTURE_READY,sample_P4_JOINT_STRUCTURE_READY,sample_P4_E_SELECTIVITY_READY,sample_P4_P_SELECTIVITY_READY,sample_P4_JOINT_SELECTIVITY_READY,p2_sample_id
2799,2024,OC2024_03830,동국대학교(WISE)_분교,동국대학교,동국대학교,분교|WISE,,분교,분교,영어영문학과,...,True,True,True,True,True,True,True,True,True,P2_STRUCTURE_READY
4301,2024,OC2024_05829,성신여자대학교,성신여자대학교,성신여자대학교,,,,unknown,교육학과,...,True,True,True,True,True,True,True,True,True,P2_STRUCTURE_READY
942,2024,OC2024_01241,경성대학교,경성대학교,경성대학교,,,,unknown,물류학전공,...,False,True,False,True,True,True,False,False,False,P2_STRUCTURE_READY
5078,2024,OC2024_06921,영남대학교,영남대학교,영남대학교,,,,unknown,음악과,...,False,True,False,True,True,True,False,False,False,P2_STRUCTURE_READY
5515,2024,OC2024_07506,이화여자대학교,이화여자대학교,이화여자대학교,,,,unknown,기후에너지시스템공학과,...,False,True,False,False,False,False,False,False,False,P2_STRUCTURE_READY
3,2024,OC2024_00004,가야대학교(김해),가야대학교,가야대학교,김해,,,unknown,귀금속주얼리학과,...,False,True,False,True,True,True,False,False,False,P2_STRUCTURE_READY
6338,2024,OC2024_08602,창신대학교,창신대학교,창신대학교,,,,unknown,식품영양학과,...,True,True,True,True,True,True,True,True,True,P2_STRUCTURE_READY
5606,2024,OC2024_07630,인제대학교,인제대학교,인제대학교,,,,unknown,인문문화학부,...,False,True,False,False,False,False,False,False,False,P2_STRUCTURE_READY
5054,2024,OC2024_06896,영남대학교,영남대학교,영남대학교,,,,unknown,생명과학과,...,True,True,True,True,True,True,True,True,True,P2_STRUCTURE_READY
2594,2024,OC2024_03531,대전대학교,대전대학교,대전대학교,,,,unknown,물류통상학과,...,True,True,True,True,True,True,True,True,True,P2_STRUCTURE_READY


<class 'pandas.DataFrame'>
RangeIndex: 7592 entries, 0 to 7591
Columns: 172 entries, analysis_year to p2_sample_id
dtypes: Float32(57), Int16(7), Int32(13), bool(18), boolean(9), category(4), float64(1), int64(1), str(24), string(38)
memory usage: 13.2 MB


...

<class 'pandas.DataFrame'>
Index: 3119 entries, 0 to 7591
Columns: 172 entries, analysis_year to p2_sample_id
dtypes: Float32(57), Int16(7), Int32(13), bool(18), boolean(9), category(4), float64(1), int64(1), str(24), string(38)
memory usage: 5.4 MB


,column,missing_n,missing_pct
0,ctx24_industry_top3_pct,3119,1.000000
1,ctx24_industry_hhi,3119,1.000000
2,health_employment_rate_pct,764,0.244950
3,employment_rate_pct,764,0.244950
4,progression_rate_pct,743,0.238217
5,vocational_college_progression_rate_pct,743,0.238217
6,university_progression_rate_pct,743,0.238217
7,graduate_school_progression_rate_pct,743,0.238217
8,domestic_progression_rate_pct,743,0.238217
9,overseas_progression_rate_pct,743,0.238217


,major_group_7,row_n,school_n
0,ART,234,82
1,EDU,365,77
2,ENG,822,103
3,HUM,366,91
4,MED,255,90
5,NAT,477,84
6,SOC,600,106


,branch,n,mean,std,min,p01,p05,p25,median,p75,p95,p99,max,zero_ratio,hundred_ratio,skewness
0,P2-S,7592,41.664440,14.373115,0.0,3.556818,22.824521,32.893457,39.572113,48.571430,69.223560,83.991481,100.0,0.008825,0.002503,0.630405
1,P2-Q,3119,40.761742,12.600778,0.0,5.000000,25.857971,33.313606,38.973850,46.521971,64.928526,80.175737,100.0,0.007374,0.001603,0.722147


,variable,has_selectivity,n,mean,median,missing_pct,level,share_no_selectivity,share_has_selectivity
0,a_rate_pct,False,4473.0,42.293884,40.219994,0.000000,NaN,NaN,NaN
1,a_rate_pct,True,3119.0,40.761742,38.973850,0.000000,NaN,NaN,NaN
2,log_enrolled_students,False,4473.0,4.638486,4.736198,0.000000,NaN,NaN,NaN
3,log_enrolled_students,True,3119.0,5.253861,5.273000,0.000000,NaN,NaN,NaN
4,log_graduates,False,4473.0,2.604549,3.178054,0.000000,NaN,NaN,NaN
5,log_graduates,True,3119.0,2.811035,3.433987,0.000000,NaN,NaN,NaN
6,student_faculty_ratio,False,4473.0,22.109516,13.166667,0.329980,NaN,NaN,NaN
7,student_faculty_ratio,True,3119.0,17.708256,14.658333,0.038153,NaN,NaN,NaN
8,major_group_7,NaN,NaN,NaN,NaN,NaN,ART,0.195171,0.075024
9,major_group_7,NaN,NaN,NaN,NaN,NaN,EDU,0.066175,0.117025


In [6]:
def audit_log_recalc(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col, source_col in [("log_enrolled_students", "enrolled_students_n"), ("log_graduates", "graduates_n")]:
        calc = np.log1p(pd.to_numeric(frame[source_col], errors="coerce"))
        stored = pd.to_numeric(frame[col], errors="coerce")
        rows.append({"column": col, "source_column": source_col, "max_abs_diff": float((stored - calc).abs().max()), "median_abs_diff": float((stored - calc).abs().median()), "status": "PASS" if (stored - calc).abs().max() < 1e-5 else "WARN"})
    return pd.DataFrame(rows)

df_log_audit = audit_log_recalc(df_d08)
display(df_log_audit)
df_log_audit.to_csv(QA_DIR / "P2_LOG_DERIVATION_AUDIT.csv", index=False)

ratio_specs = {
    "student_faculty_ratio": ("enrolled_students_n", ["fulltime_faculty_n", "nonfulltime_faculty_n"], "enrolled / (fulltime + nonfulltime faculty)"),
    "competition_ratio": ("applicants_n", ["recruitment_n"], "applicants / recruitment"),
    "admission_yield_ratio": ("admits_n", ["recruitment_n"], "admits / recruitment"),
    "admit_per_applicant_ratio": ("admits_n", ["applicants_n"], "admits / applicants"),
}
outlier_rows = []
for ratio_col, (num_col, denom_cols, formula_note) in ratio_specs.items():
    tmp = df_base[[KEY, "school_uid", "school_name_raw", "dept_name_raw", "major_group_7", "split", ratio_col, num_col] + denom_cols].copy()
    numerator = pd.to_numeric(tmp[num_col], errors="coerce")
    denominator = sum(pd.to_numeric(tmp[c], errors="coerce") for c in denom_cols)
    tmp["numerator"] = numerator
    tmp["denominator"] = denominator
    tmp["zero_denominator"] = denominator.eq(0)
    tmp["saved_ratio"] = pd.to_numeric(tmp[ratio_col], errors="coerce")
    tmp["recalculated_ratio"] = numerator / denominator.replace({0: np.nan})
    tmp["abs_diff"] = (tmp["saved_ratio"] - tmp["recalculated_ratio"]).abs()
    tmp["ratio_column"] = ratio_col
    tmp["formula_note"] = formula_note
    low = tmp.nsmallest(30, "saved_ratio", keep="all")
    high = tmp.nlargest(30, "saved_ratio", keep="all")
    outlier_rows.append(pd.concat([low.assign(tail="low30"), high.assign(tail="high30")], ignore_index=True))
df_outlier_lineage = pd.concat(outlier_rows, ignore_index=True)
display(df_outlier_lineage.head(20))
df_outlier_lineage.to_csv(QA_DIR / "P2_OUTLIER_LINEAGE_AUDIT.csv", index=False)

,column,source_column,max_abs_diff,median_abs_diff,status
0,log_enrolled_students,enrolled_students_n,4.337720e-07,8.831316e-08,PASS
1,log_graduates,graduates_n,2.383994e-07,4.347748e-08,PASS


,outcome_row_id,school_uid,school_name_raw,dept_name_raw,major_group_7,split,student_faculty_ratio,enrolled_students_n,fulltime_faculty_n,nonfulltime_faculty_n,...,abs_diff,ratio_column,formula_note,tail,competition_ratio,applicants_n,recruitment_n,admission_yield_ratio,admits_n,admit_per_applicant_ratio
0,OC2024_02854,SCH_65cc323b97e2,극동대학교,AI컴퓨터공학과,ENG,train,0.0,0,6,0,...,0.0,student_faculty_ratio,enrolled / (fulltime + nonfulltime faculty),low30,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,OC2024_02871,SCH_65cc323b97e2,극동대학교,사회복지학과,SOC,train,0.0,0,4,31,...,0.0,student_faculty_ratio,enrolled / (fulltime + nonfulltime faculty),low30,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,OC2024_02889,SCH_65cc323b97e2,극동대학교,해킹보안학과,ENG,train,0.0,0,4,14,...,0.0,student_faculty_ratio,enrolled / (fulltime + nonfulltime faculty),low30,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,OC2024_03193,SCH_a1df1d1acdd7,대구가톨릭대학교,뷰티케어전공,ART,train,0.0,0,0,1,...,0.0,student_faculty_ratio,enrolled / (fulltime + nonfulltime faculty),low30,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,OC2024_03251,SCH_a1df1d1acdd7,대구가톨릭대학교,창업경영학과,SOC,train,0.0,0,0,5,...,0.0,student_faculty_ratio,enrolled / (fulltime + nonfulltime faculty),low30,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,OC2024_03424,SCH_bcfc477eeec3,대구한의대학교,부동산개발학과,SOC,train,0.0,0,0,5,...,0.0,student_faculty_ratio,enrolled / (fulltime + nonfulltime faculty),low30,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,OC2024_03501,SCH_bcfc477eeec3,대구한의대학교,화장품학과,ENG,train,0.0,0,13,1,...,0.0,student_faculty_ratio,enrolled / (fulltime + nonfulltime faculty),low30,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,OC2024_04118,SCH_d7cf6b01676b,동신대학교,관광경영학과,SOC,val,0.0,0,3,3,...,0.0,student_faculty_ratio,enrolled / (fulltime + nonfulltime faculty),low30,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
8,OC2024_04166,SCH_d7cf6b01676b,동신대학교,호텔경영학과,SOC,val,0.0,0,3,1,...,0.0,student_faculty_ratio,enrolled / (fulltime + nonfulltime faculty),low30,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
9,OC2024_04373,SCH_854da99e21e4,동의대학교,부동산개발경영학전공,SOC,train,0.0,0,3,8,...,0.0,student_faculty_ratio,enrolled / (fulltime + nonfulltime faculty),low30,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [7]:
class DesignPrep:
    def __init__(self, categorical, numeric):
        self.categorical = list(categorical)
        self.numeric = list(numeric)
        self.categories = {}
        self.reference = {}
        self.medians = {}
        self.numeric_missing_cols = []

def fit_prep(frame: pd.DataFrame, categorical: list[str], numeric: list[str]) -> DesignPrep:
    prep = DesignPrep(categorical, numeric)
    for col in categorical:
        safe = frame[col].astype("string").fillna("__MISSING__").astype(str)
        cats = sorted(safe.dropna().unique().tolist())
        if "__OTHER__" not in cats:
            cats.append("__OTHER__")
        prep.categories[col] = cats
        prep.reference[col] = cats[0] if cats else "__MISSING__"
    for col in numeric:
        s = pd.to_numeric(frame[col], errors="coerce")
        prep.medians[col] = float(s.median()) if s.notna().any() else 0.0
        if s.isna().any():
            prep.numeric_missing_cols.append(f"{col}__missing")
    return prep

def transform_design(frame: pd.DataFrame, prep: DesignPrep, include_intercept: bool = True) -> pd.DataFrame:
    parts = []
    idx = frame.index
    if include_intercept:
        parts.append(pd.DataFrame({"Intercept": np.ones(len(frame))}, index=idx))
    for col in prep.categorical:
        raw = frame[col].astype("string").fillna("__MISSING__").astype(str)
        cats = prep.categories[col]
        raw = raw.where(raw.isin(cats), "__OTHER__")
        dummies = pd.get_dummies(pd.Categorical(raw, categories=cats), prefix=col, dtype=float)
        ref_col = f"{col}_{prep.reference[col]}"
        if ref_col in dummies.columns:
            dummies = dummies.drop(columns=[ref_col])
        dummies.index = idx
        parts.append(dummies)
    for col in prep.numeric:
        s = pd.to_numeric(frame[col], errors="coerce")
        miss = s.isna().astype(float)
        filled = s.fillna(prep.medians[col]).astype(float)
        parts.append(pd.DataFrame({col: filled}, index=idx))
        miss_col = f"{col}__missing"
        if miss_col in prep.numeric_missing_cols:
            parts.append(pd.DataFrame({miss_col: miss}, index=idx))
    X = pd.concat(parts, axis=1) if parts else pd.DataFrame(index=idx)
    X = X.astype(float)
    return X

def drop_constant_columns(X: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    constants = [c for c in X.columns if c != "Intercept" and X[c].nunique(dropna=False) <= 1]
    return X.drop(columns=constants), constants

def duplicate_columns(X: pd.DataFrame) -> list[str]:
    seen = {}
    dupes = []
    for col in X.columns:
        key = tuple(np.asarray(X[col]).round(12))
        if key in seen:
            dupes.append(f"{col}=={seen[key]}")
        else:
            seen[key] = col
    return dupes

def drop_duplicate_columns(X: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    seen = {}
    drop_cols = []
    dupes = []
    for col in X.columns:
        key = tuple(np.asarray(X[col]).round(12))
        if key in seen:
            drop_cols.append(col)
            dupes.append(f"{col}=={seen[key]}")
        else:
            seen[key] = col
    return X.drop(columns=drop_cols), dupes

def fit_spec_ols(frame: pd.DataFrame, spec: dict, track: str = "DEV"):
    dev = frame.loc[frame["split"].isin(DEV_SPLITS)].copy()
    test = frame.loc[frame["split"].eq("test")].copy()
    prep = fit_prep(dev, spec["categorical"], spec["numeric"])
    X_dev_raw = transform_design(dev, prep)
    X_dev, constant_cols = drop_constant_columns(X_dev_raw)
    X_dev, dupes = drop_duplicate_columns(X_dev)
    y_dev = pd.to_numeric(dev[TARGET], errors="coerce").astype(float)
    ok = y_dev.notna()
    X_dev = X_dev.loc[ok]
    y_dev = y_dev.loc[ok]
    groups_dev = dev.loc[ok, "school_uid"]
    rank = int(np.linalg.matrix_rank(X_dev.to_numpy())) if X_dev.shape[1] else 0
    rank_def = int(X_dev.shape[1] - rank)
    cond = float(np.linalg.cond(X_dev.to_numpy())) if X_dev.shape[1] and len(X_dev) else np.nan
    if rank_def:
        return None, prep, X_dev, y_dev, {"status": "BLOCKED_RANK", "rank": rank, "rank_deficiency": rank_def, "condition_number": cond, "constant_cols": constant_cols, "duplicate_cols": dupes}
    result = sm.OLS(y_dev, X_dev).fit(cov_type="cluster", cov_kwds={"groups": groups_dev})
    test_metrics = {}
    if len(test):
        X_test = transform_design(test, prep).reindex(columns=X_dev.columns, fill_value=0.0)
        y_test = pd.to_numeric(test[TARGET], errors="coerce")
        ok_test = y_test.notna()
        pred = result.predict(X_test.loc[ok_test])
        test_metrics = {
            "test_n": int(ok_test.sum()),
            "test_mae": float(mean_absolute_error(y_test.loc[ok_test], pred)),
            "test_rmse": float(mean_squared_error(y_test.loc[ok_test], pred) ** 0.5),
            "test_r2": float(r2_score(y_test.loc[ok_test], pred)) if ok_test.sum() > 1 else np.nan,
        }
    meta = {
        "status": "READY",
        "rank": rank,
        "rank_deficiency": rank_def,
        "condition_number": cond,
        "constant_cols": constant_cols,
        "duplicate_cols": dupes,
        "dev_n": int(len(y_dev)),
        "school_n": int(groups_dev.nunique()),
        **test_metrics,
    }
    return result, prep, X_dev, y_dev, meta

SPECS = [
    {"branch": "P2-S", "model_id": "P2-S0", "categorical": [], "numeric": [], "block_added": "intercept"},
    {"branch": "P2-S", "model_id": "P2-S1", "categorical": S0, "numeric": [], "block_added": "S0"},
    {"branch": "P2-S", "model_id": "P2-S2", "categorical": S0, "numeric": B_CORE, "block_added": "B_CORE"},
    {"branch": "P2-S", "model_id": "P2-S3", "categorical": S0, "numeric": B_CORE + B_SCALE, "block_added": "B_SCALE"},
    {"branch": "P2-S", "model_id": "P2-S4", "categorical": S0 + POLICY, "numeric": B_CORE + B_SCALE, "block_added": "POLICY"},
    {"branch": "P2-Q", "model_id": "P2-Q0", "categorical": [], "numeric": [], "block_added": "intercept"},
    {"branch": "P2-Q", "model_id": "P2-Q1", "categorical": S0, "numeric": [], "block_added": "S0"},
    {"branch": "P2-Q", "model_id": "P2-Q2", "categorical": S0 + POLICY, "numeric": B_CORE + B_SCALE, "block_added": "B_CORE_B_SCALE_POLICY"},
    {"branch": "P2-Q", "model_id": "P2-Q3", "categorical": S0 + POLICY, "numeric": B_CORE + B_SCALE + ["selectivity_proxy_pct"], "block_added": "selectivity_proxy"},
    {"branch": "P2-Q", "model_id": "P2-Q4", "categorical": S0 + POLICY, "numeric": B_CORE + B_SCALE + ["selectivity_proxy_pct", "competition_ratio"], "block_added": "competition_ratio"},
    {"branch": "P2-Q", "model_id": "P2-Q4-YIELD", "categorical": S0 + POLICY, "numeric": B_CORE + B_SCALE + ["selectivity_proxy_pct", "admission_yield_ratio"], "block_added": "admission_yield_ratio"},
    {"branch": "P2-Q", "model_id": "P2-Q4-ADMIT", "categorical": S0 + POLICY, "numeric": B_CORE + B_SCALE + ["selectivity_proxy_pct", "admit_per_applicant_ratio"], "block_added": "admit_per_applicant_ratio"},
]

baseline_feature_rows = []
for spec in SPECS:
    for col in spec["categorical"]:
        baseline_feature_rows.append({"model_id": spec["model_id"], "feature": col, "role": "categorical"})
    for col in spec["numeric"]:
        baseline_feature_rows.append({"model_id": spec["model_id"], "feature": col, "role": "numeric"})
pd.DataFrame(baseline_feature_rows).to_csv(ARTIFACTS_DIR / "P2_FEATURE_SET_BY_SPEC.csv", index=False)

In [8]:
coef_rows = []
spec_rows = []
rank_rows = []
vif_rows = []
joint_rows = []
test_rows = []
fitted_specs = {}

def add_vif_rows(model_id: str, X: pd.DataFrame):
    cols = [c for c in X.columns if c != "Intercept"]
    if not cols:
        return
    Xv = X[cols].astype(float)
    for i, col in enumerate(cols):
        try:
            vif = float(variance_inflation_factor(Xv.to_numpy(), i))
        except Exception:
            vif = np.nan
        vif_rows.append({"model_id": model_id, "encoded_feature": col, "vif": vif})

def add_joint_tests(model_id: str, result, X: pd.DataFrame, categorical: list[str]):
    for cat in categorical:
        cols = [c for c in X.columns if c.startswith(f"{cat}_")]
        if not cols:
            continue
        R = np.zeros((len(cols), len(result.params)))
        param_index = {p: i for i, p in enumerate(result.params.index)}
        for r, col in enumerate(cols):
            R[r, param_index[col]] = 1
        try:
            wt = result.wald_test(R, scalar=False)
            joint_rows.append({"model_id": model_id, "block": cat, "df": len(cols), "wald_stat": float(np.asarray(wt.statistic).ravel()[0]), "p_value": float(np.asarray(wt.pvalue).ravel()[0])})
        except Exception as exc:
            joint_rows.append({"model_id": model_id, "block": cat, "df": len(cols), "wald_stat": np.nan, "p_value": np.nan, "warning": str(exc)[:500]})

for spec in SPECS:
    frame = df_p2_structure if spec["branch"] == "P2-S" else df_p2_selectivity
    branch_status = P2_S_FEATURE_STATUS if spec["branch"] == "P2-S" else P2_Q_FEATURE_STATUS
    if branch_status != "READY":
        spec_rows.append({"model_id": spec["model_id"], "branch": spec["branch"], "status": branch_status, "reason": "feature contract blocked"})
        continue
    result, prep, X_dev, y_dev, meta = fit_spec_ols(frame, spec)
    rank_rows.append(
        {
            "model_id": spec["model_id"],
            "branch": spec["branch"],
            "raw_feature_count": len(spec["categorical"]) + len(spec["numeric"]),
            "encoded_feature_count": int(X_dev.shape[1]) if X_dev is not None else 0,
            "rank": meta.get("rank"),
            "rank_deficiency": meta.get("rank_deficiency"),
            "condition_number": meta.get("condition_number"),
            "constant_encoded_columns": "|".join(meta.get("constant_cols", [])),
            "duplicate_encoded_columns": "|".join(meta.get("duplicate_cols", [])),
            "status": meta["status"],
        }
    )
    if meta["status"] != "READY":
        spec_rows.append({"model_id": spec["model_id"], "branch": spec["branch"], "status": meta["status"], "reason": "rank deficiency"})
        continue
    fitted_specs[spec["model_id"]] = {"result": result, "prep": prep, "X": X_dev, "y": y_dev, "spec": spec, "meta": meta}
    y_sd = float(y_dev.std(ddof=1))
    conf = result.conf_int()
    for param, beta in result.params.items():
        x_sd = float(X_dev[param].std(ddof=1)) if param in X_dev.columns else np.nan
        coef_rows.append(
            {
                "model_id": spec["model_id"],
                "branch": spec["branch"],
                "term": param,
                "coefficient": float(beta),
                "school_clustered_se": float(result.bse[param]),
                "ci_low": float(conf.loc[param, 0]),
                "ci_high": float(conf.loc[param, 1]),
                "p_value": float(result.pvalues[param]),
                "standardized_beta": float(beta * x_sd / y_sd) if param != "Intercept" and y_sd > 0 and pd.notna(x_sd) else np.nan,
                "N": meta["dev_n"],
                "school_n": meta["school_n"],
                "r2": float(result.rsquared),
                "adj_r2": float(result.rsquared_adj),
                "aic": float(result.aic),
                "bic": float(result.bic),
                "condition_number": meta["condition_number"],
            }
        )
    spec_rows.append(
        {
            "model_id": spec["model_id"],
            "branch": spec["branch"],
            "status": "READY",
            "N": meta["dev_n"],
            "school_n": meta["school_n"],
            "r2": float(result.rsquared),
            "adj_r2": float(result.rsquared_adj),
            "aic": float(result.aic),
            "bic": float(result.bic),
            "condition_number": meta["condition_number"],
            "test_mae": meta.get("test_mae"),
            "test_rmse": meta.get("test_rmse"),
            "test_r2": meta.get("test_r2"),
            "block_added": spec["block_added"],
        }
    )
    if "test_mae" in meta:
        test_rows.append({"model_id": spec["model_id"], "branch": spec["branch"], "test_n": meta["test_n"], "test_mae": meta["test_mae"], "test_rmse": meta["test_rmse"], "test_r2": meta["test_r2"], "test_usage": "locked_once_after_spec_definition"})
    add_vif_rows(spec["model_id"], X_dev)
    add_joint_tests(spec["model_id"], result, X_dev, spec["categorical"])

df_model_spec = pd.DataFrame(spec_rows)
df_ols_coefficients = pd.DataFrame(coef_rows)
df_rank = pd.DataFrame(rank_rows)
df_vif = pd.DataFrame(vif_rows).sort_values(["model_id", "vif"], ascending=[True, False]).groupby("model_id").head(30).reset_index(drop=True) if vif_rows else pd.DataFrame(columns=["model_id", "encoded_feature", "vif"])
df_joint_tests = pd.DataFrame(joint_rows)
df_test_performance = pd.DataFrame(test_rows)

df_model_spec.to_csv(ARTIFACTS_DIR / "P2_MODEL_SPEC_REGISTRY.csv", index=False)
df_ols_coefficients.to_csv(ARTIFACTS_DIR / "P2_OLS_COEFFICIENTS.csv", index=False)
df_rank.to_csv(QA_DIR / "P2_DESIGN_MATRIX_RANK.csv", index=False)
df_vif.to_csv(QA_DIR / "P2_VIF_TOP30.csv", index=False)
df_joint_tests.to_csv(ARTIFACTS_DIR / "P2_JOINT_WALD_TESTS.csv", index=False)
df_test_performance.to_csv(ARTIFACTS_DIR / "P2_LOCKED_TEST_PERFORMANCE.csv", index=False)
df_test_performance.to_csv(QA_DIR / "P2_TEST_USAGE_AUDIT.csv", index=False)
display(df_model_spec)

,model_id,branch,status,N,school_n,r2,adj_r2,aic,bic,condition_number,test_mae,test_rmse,test_r2,block_added,reason
0,P2-S0,P2-S,READY,6702.0,167.0,1.110223e-16,1.110223e-16,54833.844738,54840.654899,1.000000,10.019603,13.650441,-0.002345,intercept,NaN
1,P2-S1,P2-S,READY,6702.0,167.0,8.842039e-02,8.473255e-02,54267.398071,54458.082586,273.216474,10.309273,14.174188,-0.080738,S0,NaN
2,P2-S2,P2-S,READY,6702.0,167.0,1.041557e-01,9.958712e-02,54164.700831,54403.056475,13522.850735,10.363653,14.258991,-0.093708,B_CORE,NaN
3,P2-S3,P2-S,READY,6702.0,167.0,1.263332e-01,1.216142e-01,54000.698326,54252.674293,13560.670539,10.426699,14.085459,-0.067249,B_SCALE,NaN
4,P2-S4,P2-S,READY,6702.0,167.0,1.264461e-01,1.215960e-01,54001.831575,54260.617703,13561.076066,10.432357,14.082241,-0.066762,POLICY,NaN
5,P2-Q0,P2-Q,BLOCKED_FEATURE_CONTRACT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,feature contract blocked
6,P2-Q1,P2-Q,BLOCKED_FEATURE_CONTRACT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,feature contract blocked
7,P2-Q2,P2-Q,BLOCKED_FEATURE_CONTRACT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,feature contract blocked
8,P2-Q3,P2-Q,BLOCKED_FEATURE_CONTRACT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,feature contract blocked
9,P2-Q4,P2-Q,BLOCKED_FEATURE_CONTRACT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,feature contract blocked


In [9]:
def cv_for_spec(frame: pd.DataFrame, spec: dict, max_splits: int = 5) -> dict:
    dev = frame.loc[frame["split"].isin(DEV_SPLITS)].copy()
    groups = dev["school_uid"].astype(str)
    n_splits = min(max_splits, groups.nunique())
    gkf = GroupKFold(n_splits=n_splits)
    preds = []
    ys = []
    fold_rows = []
    for fold, (tr, va) in enumerate(gkf.split(dev, groups=groups), start=1):
        train = dev.iloc[tr].copy()
        valid = dev.iloc[va].copy()
        prep = fit_prep(train, spec["categorical"], spec["numeric"])
        X_train, const = drop_constant_columns(transform_design(train, prep))
        X_train, dupes = drop_duplicate_columns(X_train)
        y_train = pd.to_numeric(train[TARGET], errors="coerce")
        ok_train = y_train.notna()
        X_train = X_train.loc[ok_train]
        y_train = y_train.loc[ok_train]
        rank = np.linalg.matrix_rank(X_train.to_numpy())
        if rank < X_train.shape[1]:
            continue
        res = sm.OLS(y_train, X_train).fit()
        X_valid = transform_design(valid, prep).reindex(columns=X_train.columns, fill_value=0.0)
        y_valid = pd.to_numeric(valid[TARGET], errors="coerce")
        ok_valid = y_valid.notna()
        pred = res.predict(X_valid.loc[ok_valid])
        preds.extend(pred.tolist())
        ys.extend(y_valid.loc[ok_valid].tolist())
        fold_rows.append({"model_id": spec["model_id"], "fold": fold, "n": int(ok_valid.sum()), "mae": float(mean_absolute_error(y_valid.loc[ok_valid], pred)), "r2": float(r2_score(y_valid.loc[ok_valid], pred)) if ok_valid.sum() > 1 else np.nan})
    return {
        "model_id": spec["model_id"],
        "cv_n": len(ys),
        "cv_mae": float(mean_absolute_error(ys, preds)) if ys else np.nan,
        "cv_rmse": float(mean_squared_error(ys, preds) ** 0.5) if ys else np.nan,
        "cv_r2": float(r2_score(ys, preds)) if len(ys) > 1 else np.nan,
        "fold_rows": fold_rows,
    }

cv_rows = []
cv_fold_rows = []
for spec in SPECS:
    if spec["model_id"] not in fitted_specs:
        continue
    frame = df_p2_structure if spec["branch"] == "P2-S" else df_p2_selectivity
    payload = cv_for_spec(frame, spec)
    fold_rows = payload.pop("fold_rows")
    cv_rows.append(payload)
    cv_fold_rows.extend(fold_rows)
df_cv = pd.DataFrame(cv_rows)
df_cv_folds = pd.DataFrame(cv_fold_rows)
df_cv.to_csv(ARTIFACTS_DIR / "P2_GROUPK_FOLD_CV_METRICS.csv", index=False)
df_cv_folds.to_csv(QA_DIR / "P2_GROUPK_FOLD_CV_FOLDS.csv", index=False)

inc_rows = []
nested_pairs = [
    ("P2-S1", "P2-S0", "S0"),
    ("P2-S2", "P2-S1", "B_CORE"),
    ("P2-S3", "P2-S2", "B_SCALE"),
    ("P2-S4", "P2-S3", "Policy"),
    ("P2-Q3", "P2-Q2", "selectivity_proxy"),
    ("P2-Q4", "P2-Q3", "competition_ratio"),
]
spec_r2 = df_model_spec.set_index("model_id")["r2"].to_dict() if "r2" in df_model_spec.columns else {}
cv_map = df_cv.set_index("model_id").to_dict("index") if len(df_cv) else {}
for new_id, old_id, block in nested_pairs:
    if new_id in spec_r2 and old_id in spec_r2:
        inc_rows.append(
            {
                "branch": "P2-S" if new_id.startswith("P2-S") else "P2-Q",
                "from_model": old_id,
                "to_model": new_id,
                "block_added": block,
                "delta_r2_dev": spec_r2[new_id] - spec_r2[old_id],
                "delta_cv_mae": cv_map.get(old_id, {}).get("cv_mae", np.nan) - cv_map.get(new_id, {}).get("cv_mae", np.nan),
                "delta_cv_r2": cv_map.get(new_id, {}).get("cv_r2", np.nan) - cv_map.get(old_id, {}).get("cv_r2", np.nan),
                "status": "READY",
            }
        )
    else:
        inc_rows.append({"branch": "P2-S" if new_id.startswith("P2-S") else "P2-Q", "from_model": old_id, "to_model": new_id, "block_added": block, "status": "BLOCKED_FEATURE_CONTRACT"})
df_incremental = pd.DataFrame(inc_rows)
display(df_incremental)
df_incremental.to_csv(ARTIFACTS_DIR / "P2_BLOCK_INCREMENTAL_VALUE.csv", index=False)

,branch,from_model,to_model,block_added,delta_r2_dev,delta_cv_mae,delta_cv_r2,status
0,P2-S,P2-S0,P2-S1,S0,0.088420,0.154054,0.030860,READY
1,P2-S,P2-S1,P2-S2,B_CORE,0.015735,0.057888,0.012187,READY
2,P2-S,P2-S2,P2-S3,B_SCALE,0.022177,0.190278,0.028122,READY
3,P2-S,P2-S3,P2-S4,Policy,0.000113,-0.097367,-0.013531,READY
4,P2-Q,P2-Q2,P2-Q3,selectivity_proxy,NaN,NaN,NaN,READY
5,P2-Q,P2-Q3,P2-Q4,competition_ratio,NaN,NaN,NaN,READY


In [10]:
gam_rows = []
gam_candidates = [
    ("student_faculty_ratio", "P2-S4", "P2-S"),
    ("log_enrolled_students", "P2-S4", "P2-S"),
    ("log_graduates", "P2-S4", "P2-S"),
    ("selectivity_proxy_pct", "P2-Q4", "P2-Q"),
    ("competition_ratio", "P2-Q4", "P2-Q"),
]

def spline_compare(var: str, model_id: str):
    fit = fitted_specs[model_id]
    spec = fit["spec"]
    frame = df_p2_structure if spec["branch"] == "P2-S" else df_p2_selectivity
    dev = frame.loc[frame["split"].isin(DEV_SPLITS)].copy()
    prep = fit_prep(dev, spec["categorical"], spec["numeric"])
    X_base, _ = drop_constant_columns(transform_design(dev, prep))
    X_base, _ = drop_duplicate_columns(X_base)
    y = pd.to_numeric(dev[TARGET], errors="coerce")
    ok = y.notna()
    X_base = X_base.loc[ok]
    y = y.loc[ok]
    if var not in X_base.columns:
        return {"variable": var, "model_id": model_id, "status": "NOT_ESTIMABLE", "reason": "variable not in base design"}
    base_res = sm.OLS(y, X_base).fit()
    spline = patsy.dmatrix("bs(x, df=4, include_intercept=False)", {"x": X_base[var]}, return_type="dataframe")
    spline.columns = [f"{var}_spline_{i}" for i in range(spline.shape[1])]
    spline.index = X_base.index
    X_spline = pd.concat([X_base.drop(columns=[var]), spline], axis=1)
    X_spline, _ = drop_constant_columns(X_spline)
    X_spline, _ = drop_duplicate_columns(X_spline)
    rank = np.linalg.matrix_rank(X_spline.to_numpy())
    if rank < X_spline.shape[1]:
        return {"variable": var, "model_id": model_id, "status": "BLOCKED_RANK", "rank": int(rank), "encoded_n": X_spline.shape[1]}
    spline_res = sm.OLS(y, X_spline).fit()
    grid = np.linspace(float(X_base[var].quantile(.01)), float(X_base[var].quantile(.99)), 100)
    template = X_base.median(numeric_only=True).to_frame().T.loc[np.repeat(X_base.index[:1], len(grid))].copy()
    template.index = range(len(grid))
    template[var] = grid
    spline_grid = patsy.dmatrix("bs(x, df=4, include_intercept=False)", {"x": template[var]}, return_type="dataframe")
    spline_grid.columns = [f"{var}_spline_{i}" for i in range(spline_grid.shape[1])]
    Xg = pd.concat([template.drop(columns=[var]), spline_grid], axis=1).reindex(columns=X_spline.columns, fill_value=0.0)
    pred = spline_res.get_prediction(Xg).summary_frame(alpha=0.05)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(grid, pred["mean"], label="spline partial")
    ax.fill_between(grid, pred["mean_ci_lower"], pred["mean_ci_upper"], alpha=0.2)
    ax.set_xlabel(var)
    ax.set_ylabel("A rate pct")
    ax.set_title(f"{model_id}: spline check for {var}")
    ax2 = ax.twinx()
    ax2.hist(X_base[var], bins=30, alpha=0.15)
    ax2.set_ylabel("density count")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"P2_GAM_{model_id}_{var}.png", dpi=160)
    plt.close(fig)
    return {
        "variable": var,
        "model_id": model_id,
        "status": "READY",
        "linear_aic": float(base_res.aic),
        "spline_aic": float(spline_res.aic),
        "delta_aic_spline_minus_linear": float(spline_res.aic - base_res.aic),
        "effective_degrees_of_freedom": int(spline.shape[1]),
        "figure": f"P2_GAM_{model_id}_{var}.png",
    }

for var, model_id, branch in gam_candidates:
    if model_id not in fitted_specs:
        gam_rows.append({"variable": var, "model_id": model_id, "branch": branch, "status": "BLOCKED_FEATURE_CONTRACT"})
    else:
        row = spline_compare(var, model_id)
        row["branch"] = branch
        gam_rows.append(row)

df_gam = pd.DataFrame(gam_rows)
display(df_gam)
df_gam.to_csv(ARTIFACTS_DIR / "P2_GAM_SPLINE_COMPARISON.csv", index=False)
P2_GAM_STATUS = "READY_WITH_WARNINGS" if (df_gam["status"] != "READY").any() else "READY"

,variable,model_id,status,linear_aic,spline_aic,delta_aic_spline_minus_linear,effective_degrees_of_freedom,figure,branch
0,student_faculty_ratio,P2-S4,READY,54001.831575,53978.267455,-23.564119,5.0,P2_GAM_P2-S4_student_faculty_ratio.png,P2-S
1,log_enrolled_students,P2-S4,READY,54001.831575,54001.899359,0.067785,5.0,P2_GAM_P2-S4_log_enrolled_students.png,P2-S
2,log_graduates,P2-S4,READY,54001.831575,53980.418282,-21.413293,5.0,P2_GAM_P2-S4_log_graduates.png,P2-S
3,selectivity_proxy_pct,P2-Q4,BLOCKED_FEATURE_CONTRACT,NaN,NaN,NaN,NaN,NaN,P2-Q
4,competition_ratio,P2-Q4,BLOCKED_FEATURE_CONTRACT,NaN,NaN,NaN,NaN,NaN,P2-Q


In [11]:
mixed_rows = []
fe_rows = []
fractional_rows = []

if "P2-S4" in fitted_specs:
    fit = fitted_specs["P2-S4"]
    dev = df_p2_structure.loc[df_p2_structure["split"].isin(DEV_SPLITS)].copy()
    y = pd.to_numeric(dev[TARGET], errors="coerce")
    ok = y.notna()
    dev = dev.loc[ok].copy()
    y = y.loc[ok]
    groups = dev["school_uid"].astype(str)
    try:
        null_exog = pd.DataFrame({"Intercept": np.ones(len(dev))}, index=dev.index)
        null_mod = sm.MixedLM(y, null_exog, groups=groups)
        null_res = null_mod.fit(reml=False, method="lbfgs", maxiter=200, disp=False)
        prep = fit["prep"]
        X_adj = transform_design(dev, prep).reindex(columns=fit["X"].columns, fill_value=0.0)
        adj_mod = sm.MixedLM(y, X_adj, groups=groups)
        try:
            adj_res = adj_mod.fit(reml=False, method="lbfgs", maxiter=200, disp=False)
            adj_status = "READY"
        except Exception:
            adj_res = adj_mod.fit(reml=False, method="powell", maxiter=200, disp=False)
            adj_status = "MODEL_CONVERGENCE_WARNING"
        for label, res, status in [("NULL", null_res, "READY"), ("ADJUSTED_S4", adj_res, adj_status)]:
            school_var = float(res.cov_re.iloc[0, 0]) if hasattr(res, "cov_re") and res.cov_re.size else np.nan
            resid_var = float(res.scale)
            icc = school_var / (school_var + resid_var) if pd.notna(school_var) and (school_var + resid_var) > 0 else np.nan
            mixed_rows.append({"model": label, "school_variance": school_var, "residual_variance": resid_var, "icc": icc, "log_likelihood": float(res.llf), "aic": float(res.aic), "bic": float(res.bic), "converged": bool(getattr(res, "converged", False)), "status": status if 0 <= icc <= 1 else "MODEL_CONVERGENCE_WARNING"})
    except Exception as exc:
        mixed_rows.append({"model": "MIXEDLM", "status": "MODEL_CONVERGENCE_WARNING", "error": str(exc)[:1000]})

    X = fit["X"].copy()
    candidate_cols = [c for c in X.columns if c != "Intercept"]
    within_cols = []
    for col in candidate_cols:
        varying = X[[col]].join(dev["school_uid"]).groupby("school_uid")[col].nunique(dropna=False).gt(1).any()
        if varying:
            within_cols.append(col)
    school_dummies = pd.get_dummies(dev["school_uid"].astype(str), prefix="school", drop_first=True, dtype=float)
    X_fe = pd.concat([pd.DataFrame({"Intercept": np.ones(len(dev))}, index=dev.index), school_dummies.set_index(dev.index), X[within_cols]], axis=1)
    X_fe, constants = drop_constant_columns(X_fe)
    try:
        fe_res = sm.OLS(y, X_fe).fit(cov_type="cluster", cov_kwds={"groups": groups})
        conf = fe_res.conf_int()
        for col in within_cols:
            fe_rows.append({"term": col, "coefficient": float(fe_res.params[col]), "se": float(fe_res.bse[col]), "ci_low": float(conf.loc[col, 0]), "ci_high": float(conf.loc[col, 1]), "p_value": float(fe_res.pvalues[col]), "status": "READY"})
    except Exception as exc:
        fe_rows.append({"term": "SCHOOL_FE", "status": "MODEL_CONVERGENCE_WARNING", "error": str(exc)[:1000]})

    try:
        y_prop = y / 100.0
        glm_res = sm.GLM(y_prop, X, family=sm.families.Binomial()).fit(cov_type="cluster", cov_kwds={"groups": groups})
        conf = glm_res.conf_int()
        for col in glm_res.params.index:
            fractional_rows.append({"term": col, "coefficient": float(glm_res.params[col]), "se": float(glm_res.bse[col]), "ci_low": float(conf.loc[col, 0]), "ci_high": float(conf.loc[col, 1]), "p_value": float(glm_res.pvalues[col]), "status": "READY", "converged": bool(getattr(glm_res, "converged", True))})
    except Exception as exc:
        fractional_rows.append({"term": "FRACTIONAL_LOGIT", "status": "MODEL_CONVERGENCE_WARNING", "error": str(exc)[:1000]})
else:
    mixed_rows.append({"model": "MIXEDLM", "status": "NOT_ESTIMABLE", "error": "P2-S4 not available"})
    fe_rows.append({"term": "SCHOOL_FE", "status": "NOT_ESTIMABLE", "error": "P2-S4 not available"})
    fractional_rows.append({"term": "FRACTIONAL_LOGIT", "status": "NOT_ESTIMABLE", "error": "P2-S4 not available"})

df_mixed = pd.DataFrame(mixed_rows)
df_fe = pd.DataFrame(fe_rows)
df_fractional = pd.DataFrame(fractional_rows)
df_mixed.to_csv(ARTIFACTS_DIR / "P2_MIXEDLM_VARIANCE_COMPONENTS.csv", index=False)
df_fe.to_csv(ARTIFACTS_DIR / "P2_SCHOOL_FE_SENSITIVITY.csv", index=False)
df_fractional.to_csv(ARTIFACTS_DIR / "P2_FRACTIONAL_LOGIT_SENSITIVITY.csv", index=False)
display(df_mixed)
display(df_fe.head(20))
display(df_fractional.head(20))
def model_component_status(df: pd.DataFrame) -> str:
    if df.empty or "status" not in df.columns:
        return "NOT_ESTIMABLE"
    statuses = set(df["status"].dropna().astype(str))
    if any("WARNING" in s for s in statuses):
        return "MODEL_CONVERGENCE_WARNING"
    if "READY" in statuses:
        return "READY"
    if "NOT_ESTIMABLE" in statuses:
        return "NOT_ESTIMABLE"
    return "MODEL_CONVERGENCE_WARNING"

P2_MIXEDLM_STATUS = model_component_status(df_mixed)
P2_SCHOOL_FE_STATUS = model_component_status(df_fe)
P2_FRACTIONAL_STATUS = model_component_status(df_fractional)

/home/sieg/projects-wsl/SBS_dataScience/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/home/sieg/projects-wsl/SBS_dataScience/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/home/sieg/projects-wsl/SBS_dataScience/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/home/sieg/projects-wsl/SBS_dataScience/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2245: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/home/sieg/projects-wsl/SBS_dataScience/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.p

,model,school_variance,residual_variance,icc,log_likelihood,aic,bic,converged,status
0,NULL,0.000000,141.098572,0.000000,inf,-inf,-inf,True,READY
1,ADJUSTED_S4,66.480558,130.080932,0.338218,-2.604509e+04,5.217018e+04,5.244259e+04,True,MODEL_CONVERGENCE_WARNING


,term,coefficient,se,ci_low,ci_high,p_value,status
0,major_group_7_EDU,-0.718348,0.989017,-2.656785,1.220088,4.676389e-01,READY
1,major_group_7_ENG,-3.067591,0.943703,-4.917214,-1.217967,1.151658e-03,READY
2,major_group_7_HUM,-3.528184,0.882547,-5.257943,-1.798424,6.395266e-05,READY
3,major_group_7_MED,-5.336723,1.113617,-7.519372,-3.154075,1.649259e-06,READY
4,major_group_7_NAT,-5.522195,0.986425,-7.455553,-3.588837,2.166026e-08,READY
5,major_group_7_SOC,-5.305231,0.910460,-7.089699,-3.520763,5.643907e-09,READY
6,region_sido_경기,-2.903358,0.727394,-4.329023,-1.477693,6.566951e-05,READY
7,region_sido_경남,-22.827827,1.016536,-24.820202,-20.835453,1.106094e-111,READY
8,region_sido_경북,2.852995,1.728826,-0.535442,6.241432,9.889183e-02,READY
9,region_sido_광주,3.347219,3.510641,-3.533512,10.227950,3.403624e-01,READY


,term,coefficient,se,ci_low,ci_high,p_value,status,converged
0,Intercept,0.074767,0.140348,-0.200309,0.349844,5.942215e-01,READY,True
1,major_group_7_EDU,-0.006728,0.064762,-0.133660,0.120204,9.172587e-01,READY,True
2,major_group_7_ENG,-0.098328,0.050114,-0.196548,-0.000107,4.975102e-02,READY,True
3,major_group_7_HUM,-0.091246,0.054490,-0.198045,0.015553,9.402602e-02,READY,True
4,major_group_7_MED,-0.135463,0.053827,-0.240962,-0.029964,1.184794e-02,READY,True
5,major_group_7_NAT,-0.157373,0.048078,-0.251605,-0.063141,1.063163e-03,READY,True
6,major_group_7_SOC,-0.197597,0.044673,-0.285155,-0.110039,9.727324e-06,READY,True
7,school_type_교육대학,-0.778676,0.133653,-1.040631,-0.516721,5.673613e-09,READY,True
8,school_type_대학교,-0.114418,0.088780,-0.288424,0.059587,1.974726e-01,READY,True
9,school_type_방송통신대학교,-0.686978,0.230126,-1.138016,-0.235940,2.833633e-03,READY,True


In [12]:
diagnostics = []
for _, row in df_model_spec.iterrows():
    flags = []
    if row["status"] != "READY":
        flags.append(row["status"])
    if pd.notna(row.get("condition_number", np.nan)) and row.get("condition_number", 0) > 1e5:
        flags.append("HIGH_CONDITION_NUMBER")
    diagnostics.append({"model_id": row["model_id"], "branch": row["branch"], "diagnostic_flag": " | ".join(flags), "status": row["status"]})
df_diagnostics = pd.DataFrame(diagnostics)
df_diagnostics.to_csv(QA_DIR / "P2_MODEL_DIAGNOSTICS.csv", index=False)

handoff = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "target": TARGET,
    "target_unit": "percentage_point",
    "strict_d08_path": rel(INPUTS["strict_d08"]),
    "strict_d08_sha256": sha256_file(INPUTS["strict_d08"]),
    "samples": {
        "P2_STRUCTURE_READY": sample_registry_row("P2_STRUCTURE_READY"),
        "P2_SELECTIVITY_READY": sample_registry_row("P2_SELECTIVITY_READY"),
    },
    "p2_s_feature_status": P2_S_FEATURE_STATUS,
    "p2_q_feature_status": P2_Q_FEATURE_STATUS,
    "p2_s_final_spec": {"model_id": "P2-S4", "categorical": S0 + POLICY, "numeric": B_CORE + B_SCALE},
    "p2_q_final_spec": {"model_id": "P2-Q4", "categorical": S0 + POLICY, "numeric": B_CORE + B_SCALE + Q_PRIMARY, "status": P2_Q_FEATURE_STATUS},
    "preprocessing": {
        "categorical_missing": "__MISSING__",
        "categorical_unseen": "__OTHER__",
        "coding": "treatment/drop reference",
        "numeric_imputation": "development median",
        "numeric_missing_indicator": "only when missing observed in development fit",
        "standardization": "available for prediction/standardized coefficient track",
    },
}
handoff_path = ARTIFACTS_DIR / "P2_TO_P3_FEATURE_MATRIX_SCHEMA.json"
handoff_path.write_text(json.dumps(handoff, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

interpretation_guardrails = pd.DataFrame(
    [
        {"rule": "No causal claim", "text": "P2 coefficients are conditional associations in strict 2024 department data."},
        {"rule": "Q branch scope", "text": "P2-Q applies only to strict rows with observed selectivity if feature contract allows Q predictors."},
        {"rule": "No test-driven specification", "text": "Locked test metrics are reported once after specifications are defined."},
        {"rule": "No leakage", "text": "Employment, progression, context, identifiers, and review metadata are excluded from model features."},
    ]
)
interpretation_guardrails.to_csv(REPORTS_DIR / "P2_INTERPRETATION_GUARDRAILS.csv", index=False)

P2_S_OLS_STATUS = "READY" if "P2-S4" in fitted_specs else P2_S_FEATURE_STATUS
P2_Q_OLS_STATUS = "READY" if "P2-Q4" in fitted_specs else P2_Q_FEATURE_STATUS
P2_TEST_STATUS = "READY" if len(df_test_performance) else "NOT_ESTIMABLE"
P2_TO_P3_HANDOFF_STATUS = "READY" if handoff_path.exists() else "NOT_ESTIMABLE"
P2_OVERALL_STATUS = "READY_WITH_WARNINGS" if P2_Q_OLS_STATUS != "READY" else "READY"

status_payload = {
    **EXECUTION_ENV,
    "P2_INPUT_STATUS": P2_INPUT_STATUS,
    "P2_SAMPLE_STATUS": P2_SAMPLE_STATUS,
    "P2_FEATURE_CONTRACT_STATUS": P2_FEATURE_CONTRACT_STATUS,
    "P2_S_OLS_STATUS": P2_S_OLS_STATUS,
    "P2_Q_OLS_STATUS": P2_Q_OLS_STATUS,
    "P2_GAM_STATUS": P2_GAM_STATUS,
    "P2_MIXEDLM_STATUS": P2_MIXEDLM_STATUS,
    "P2_SCHOOL_FE_STATUS": P2_SCHOOL_FE_STATUS,
    "P2_FRACTIONAL_STATUS": P2_FRACTIONAL_STATUS,
    "P2_TEST_STATUS": P2_TEST_STATUS,
    "P2_TO_P3_HANDOFF_STATUS": P2_TO_P3_HANDOFF_STATUS,
    "P2_OVERALL_STATUS": P2_OVERALL_STATUS,
    "strict_d08_path": rel(INPUTS["strict_d08"]),
    "strict_d08_shape": file_shape(INPUTS["strict_d08"]),
    "strict_d08_sha256": sha256_file(INPUTS["strict_d08"]),
    "p2_s_sample": df_sample_audit.iloc[0].to_dict(),
    "p2_q_sample": df_sample_audit.iloc[1].to_dict(),
}

def markdown_table(df: pd.DataFrame) -> str:
    if df is None or df.empty:
        return "_empty_"
    out = df.copy().astype(object).where(pd.notna(df), "")
    cols = [str(c) for c in out.columns]
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for _, row in out.iterrows():
        lines.append("| " + " | ".join(str(row[c]).replace("|", "/") for c in out.columns) + " |")
    return "\n".join(lines)

report_lines = [
    "# P2 Grade Formation Strict Report",
    "",
    "## 1. 연구질문",
    "strict-clean 2024 학과 데이터에서 대학·학과별 A비율이 전공, 학교·캠퍼스 조건, 학과구조, 입결·선발력, 성적제도와 어떤 조건부 관계를 갖는지 추정했다.",
    "",
    "## 2. 데이터와 표본",
    f"- strict D08: `{rel(INPUTS['strict_d08'])}`",
    f"- shape: `{file_shape(INPUTS['strict_d08'])}`",
    f"- SHA256: `{sha256_file(INPUTS['strict_d08'])}`",
    f"- P2-S: N={len(df_p2_structure):,}, school_n={df_p2_structure['school_uid'].nunique():,}",
    f"- P2-Q: N={len(df_p2_selectivity):,}, school_n={df_p2_selectivity['school_uid'].nunique():,}",
    "",
    "## 3. Feature contract",
    f"- P2-S feature status: `{P2_S_FEATURE_STATUS}`",
    f"- P2-Q feature status: `{P2_Q_FEATURE_STATUS}`",
    "Feature diff는 `qa/P2_FEATURE_CONTRACT_DIFF.csv`에 저장했다.",
    "",
    "## 4. OLS nested models",
    markdown_table(df_model_spec),
    "",
    "## 5. Block incremental value",
    markdown_table(df_incremental),
    "",
    "## 6. GAM/spline",
    markdown_table(df_gam),
    "",
    "## 7. MixedLM ICC",
    markdown_table(df_mixed),
    "",
    "## 8. School FE sensitivity",
    "School fixed-effect sensitivity coefficients are saved in `artifacts/P2_SCHOOL_FE_SENSITIVITY.csv`.",
    "",
    "## 9. Fractional sensitivity",
    "Fractional logit sensitivity coefficients are saved in `artifacts/P2_FRACTIONAL_LOGIT_SENSITIVITY.csv`.",
    "",
    "## 10. Locked test",
    markdown_table(df_test_performance),
    "",
    "## 11. P3 handoff",
    f"`{rel(handoff_path)}`",
]
(REPORTS_DIR / "P2_GRADE_FORMATION_REPORT.md").write_text("\n".join(report_lines), encoding="utf-8")
(REPORTS_DIR / "P2_GRADE_FORMATION_STATUS.json").write_text(json.dumps(status_payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
display(pd.DataFrame([status_payload]))

,python,platform,pandas,numpy,statsmodels,working_directory,git_commit,execution_timestamp_utc,notebook_path,output_root,...,P2_SCHOOL_FE_STATUS,P2_FRACTIONAL_STATUS,P2_TEST_STATUS,P2_TO_P3_HANDOFF_STATUS,P2_OVERALL_STATUS,strict_d08_path,strict_d08_shape,strict_d08_sha256,p2_s_sample,p2_q_sample
0,3.12.3,Linux-6.18.33.2-microsoft-standard-WSL2-x86_64...,3.0.3,2.5.0,0.14.6,/home/sieg/projects-wsl/SBS_dataScience,5b1a3d54266d881a839ad9a3cec750da66e94bc7,2026-07-13T06:29:22.053715+00:00,/home/sieg/projects-wsl/SBS_dataScience/workbo...,/home/sieg/projects-wsl/SBS_dataScience/workbo...,...,READY,READY,READY,READY,READY_WITH_WARNINGS,workbook/p2/p2_4/source_eda/strict_clean_v1/ma...,"(7592, 151)",5f56e375fd1c0474a5e55652859ae007e2f45becd6d335...,"{'sample_id': 'P2_STRUCTURE_READY', 'row_n': 7...","{'sample_id': 'P2_SELECTIVITY_READY', 'row_n':..."


In [13]:
def artifact_record(path: Path) -> dict:
    stat = path.stat()
    shape = None
    try:
        shape = file_shape(path)
    except Exception:
        shape = None
    return {
        "path": rel(path),
        "shape": shape,
        "size_bytes": int(stat.st_size),
        "sha256": sha256_file(path),
        "mtime": datetime.fromtimestamp(stat.st_mtime).isoformat(),
    }

output_files = sorted([p for p in OUTPUT_ROOT.rglob("*") if p.is_file() and p.name != "P2_OUTPUT_MANIFEST.json"])
manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "output_root": str(OUTPUT_ROOT),
    "notebook_path": str(NOTEBOOK_PATH),
    "lineage_inputs": [artifact_record(path) for path in INPUTS.values()],
    "outputs": [artifact_record(path) for path in output_files],
    "required_hashes": {
        "strict_d08_sha256": sha256_file(INPUTS["strict_d08"]),
        "sample_registry_sha256": sha256_file(INPUTS["phase_sample_registry"]),
        "structure_frame_sha256": sha256_file(DATA_DIR / "P2_STRUCTURE_FRAME.parquet"),
        "selectivity_frame_sha256": sha256_file(DATA_DIR / "P2_SELECTIVITY_FRAME.parquet"),
        "ols_coefficients_sha256": sha256_file(ARTIFACTS_DIR / "P2_OLS_COEFFICIENTS.csv"),
        "handoff_sha256": sha256_file(ARTIFACTS_DIR / "P2_TO_P3_FEATURE_MATRIX_SCHEMA.json"),
        "notebook_sha256": sha256_file(NOTEBOOK_PATH) if NOTEBOOK_PATH.exists() else None,
    },
}
manifest_path = LOGS_DIR / "P2_OUTPUT_MANIFEST.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("manifest:", rel(manifest_path))
print("outputs:", len(output_files))

manifest: workbook/p2/p2_4/p2_grade_formation_v1_strict/logs/P2_OUTPUT_MANIFEST.json
outputs: 46
